# 02 Business Analysis


In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv('../data/cleaned/ecommerce_cleaned.csv', parse_dates=['order_date', 'signup_date', 'ship_date', 'delivery_date'])
df.head()


In [ ]:
# Executive KPIs
completed = df[df['order_status'] == 'Completed'].copy()

kpis = {
    'revenue': completed['revenue'].sum(),
    'orders': completed['order_id'].nunique(),
    'customers': completed['customer_id'].nunique(),
    'average_order_value': completed['revenue'].sum() / completed['order_id'].nunique(),
    'return_rate_pct': 100 * completed['return_flag'].sum() / completed['order_id'].nunique(),
    'on_time_delivery_rate_pct': 100 * (completed['delivery_performance'] == 'On Time').mean(),
}
kpis


In [ ]:
# Revenue by month
monthly_revenue = (
    completed.groupby('order_month', as_index=False)
    .agg(revenue=('revenue', 'sum'),
         orders=('order_id', 'nunique'),
         customers=('customer_id', 'nunique'))
    .sort_values('order_month')
)
monthly_revenue


In [ ]:
# Revenue and margin by category
category_perf = (
    completed.groupby('category', as_index=False)
    .agg(revenue=('revenue', 'sum'),
         gross_margin=('gross_margin', 'sum'),
         return_rate=('return_flag', 'mean'))
    .sort_values('revenue', ascending=False)
)
category_perf['return_rate_pct'] = category_perf['return_rate'] * 100
category_perf


In [ ]:
# Customer lifetime value
clv = (
    completed.groupby(['customer_id', 'name', 'city'], as_index=False)
    .agg(total_orders=('order_id', 'nunique'),
         lifetime_value=('revenue', 'sum'),
         last_order_date=('order_date', 'max'))
    .sort_values('lifetime_value', ascending=False)
)
clv.head(10)


In [ ]:
# Churn-risk customers: >45 days since last purchase relative to max order date
max_date = completed['order_date'].max()
clv['days_since_last_order'] = (max_date - clv['last_order_date']).dt.days
churn_risk = clv[clv['days_since_last_order'] > 45].sort_values('days_since_last_order', ascending=False)
churn_risk


In [ ]:
# Simple cohort table
first_order = completed.groupby('customer_id')['order_date'].min().rename('first_order_date')
cohort_df = completed[['customer_id', 'order_date']].drop_duplicates().merge(first_order, on='customer_id')
cohort_df['cohort_month'] = cohort_df['first_order_date'].dt.to_period('M').astype(str)
cohort_df['activity_month'] = cohort_df['order_date'].dt.to_period('M').astype(str)
cohort_df['cohort_index'] = (
    (cohort_df['order_date'].dt.year - cohort_df['first_order_date'].dt.year) * 12 +
    (cohort_df['order_date'].dt.month - cohort_df['first_order_date'].dt.month)
)

cohort_counts = cohort_df.groupby(['cohort_month', 'cohort_index'])['customer_id'].nunique().unstack(fill_value=0)
cohort_size = cohort_counts.iloc[:, 0]
retention = cohort_counts.divide(cohort_size, axis=0).round(3)
retention
